In [ ]:
import csv
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
import copy
import itertools
from sklearn.preprocessing import MinMaxScaler


In [ ]:
class Row:
    def __init__(self, row):
        self.depth = int(row[0])
        self.nitrite = float(row[1])
        self.nitrate_nitrite = float(row[2])
        self.ammonia = float(row[3])
        self.silicate = float(row[4])
        self.phosphate = float(row[5])
    def __repr__(self):
        return f"Row({self.depth}, {self.nitrite}, {self.nitrate_nitrite}, {self.ammonia}, {self.silicate}, {self.phosphate})"

def filter_values(data: list[Row], column):
    iqr = np.percentile([getattr(row, column) for row in data], 75) - np.percentile([getattr(row, column) for row in data], 25)
    lower_bound = np.percentile([getattr(row, column) for row in data], 25) - 1.5 * iqr
    upper_bound = np.percentile([getattr(row, column) for row in data], 75) + 1.5 * iqr
    for row in data:
        current_value = getattr(row, column)
        setattr(row, column, current_value if lower_bound <= current_value <= upper_bound else -1)
    return data

In [ ]:
raw_data = pd.read_csv(Path("e1_nutrients.csv")).values.tolist()

numeric_data = [[float(row[i]) for i in range(1, 6)] for row in raw_data]
scaled_data = MinMaxScaler().fit_transform(numeric_data)
rows = [Row([int(raw_data[i][0])] + list(scaled_data[i])) for i in range(len(raw_data))]
rows_clone = copy.deepcopy(rows)

depths = set([row.depth for row in rows_clone])
columns = ["nitrite", "nitrate_nitrite", "ammonia", "silicate", "phosphate"]
raw_data

In [ ]:

depth_groups = [[row for row in rows_clone if row.depth == x] for x in depths]

for group in depth_groups:
    group_filtered = group
    for column in columns:
        group_filtered = filter_values(group_filtered, column)
depth_groups = list(itertools.chain.from_iterable(depth_groups))

In [ ]:
for column in columns:
    plt.figure(figsize=(6, 8))
    plt.plot([row.depth for row in depth_groups], [getattr(row, column) for row in depth_groups], 'o', markersize=5)
    plt.plot([row.depth+1 for row in rows], [getattr(row, column) for row in rows], 'x', markersize=5, alpha=0.5)
    plt.xlabel("Depth")
    plt.ylabel(column.capitalize())
    plt.title(f"{column.capitalize()} vs Depth for Filtered Data")
    plt.show()

# plt.boxplot([[getattr(row, "nitrate_nitrite") for row in depth_groups if row.depth == x] for x in depths], tick_labels=depths)